# Custom Datasets

So far we've used datasets that PyTorch ships (`MNIST`, `CIFAR-10`) or created tensors in memory.
Real projects load data from disk — folders of images, CSV files, audio clips.

This notebook covers two approaches:
1. **`datasets.ImageFolder`** — zero-code loader when your images are organised by class folder
2. **Custom `Dataset` class** — full control for any file format or on-the-fly processing

In [ ]:
import os
import pathlib
import random
import shutil

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import v2
from PIL import Image
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Set up a folder structure for image data

`ImageFolder` expects this layout — one sub-folder per class:

```
data/
  train/
    cats/   image1.jpg  image2.jpg ...
    dogs/   image1.jpg  image2.jpg ...
  test/
    cats/   ...
    dogs/   ...
```

We'll generate solid-colour synthetic images to simulate this without downloading anything.

In [ ]:
def make_synthetic_dataset(root='data/synthetic', n_per_class=50, img_size=64):
    """Create RGB images of solid colours as stand-ins for real images."""
    classes = {'red': (220, 50, 50), 'green': (50, 180, 50), 'blue': (50, 50, 220)}
    for split in ('train', 'test'):
        for cls, colour in classes.items():
            folder = pathlib.Path(root) / split / cls
            folder.mkdir(parents=True, exist_ok=True)
            count = n_per_class if split == 'train' else n_per_class // 5
            for i in range(count):
                # Add slight noise so images aren't identical
                pixel = tuple(max(0, min(255, c + random.randint(-20, 20))) for c in colour)
                img = Image.new('RGB', (img_size, img_size), pixel)
                img.save(folder / f'{cls}_{i:03d}.jpg')
    print(f'Created dataset at {root}')

make_synthetic_dataset()

# Inspect the structure
for path in sorted(pathlib.Path('data/synthetic').rglob('*.jpg'))[:6]:
    print(path)

## 2. `datasets.ImageFolder` — zero-code loading

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = datasets.ImageFolder(root='data/synthetic/train', transform=transform)
test_ds  = datasets.ImageFolder(root='data/synthetic/test',  transform=transform)

print('Classes:', train_ds.classes)          # ['blue', 'green', 'red'] — alphabetical
print('Class → index:', train_ds.class_to_idx)
print('Train samples:', len(train_ds))

# Each item is (tensor, int_label)
img, label = train_ds[0]
print(f'Image shape: {img.shape}  Label: {label} ({train_ds.classes[label]})')

In [ ]:
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=16)

xb, yb = next(iter(train_loader))
print('Batch shape:', xb.shape, '  Labels:', yb[:8].tolist())

## 3. Custom `Dataset` — full control

Use when:
- Labels live in a CSV, not folder names
- You need on-the-fly processing (audio, medical images, text+image pairs)
- You want lazy loading (load from disk only when the item is requested)

Three required methods: `__init__`, `__len__`, `__getitem__`.

In [ ]:
import pandas as pd

# First, create a CSV of (filepath, label) pairs to simulate a real annotation file
rows = []
for path in pathlib.Path('data/synthetic/train').rglob('*.jpg'):
    rows.append({'path': str(path), 'label': path.parent.name})
df = pd.DataFrame(rows)
df['label_idx'] = df['label'].map({'blue': 0, 'green': 1, 'red': 2})
df.to_csv('data/synthetic/train.csv', index=False)
print(df.head())

In [ ]:
class ImageCSVDataset(Dataset):
    """Loads images listed in a CSV with columns: path, label_idx."""

    def __init__(self, csv_path, transform=None):
        self.df        = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')  # always convert — avoids grayscale surprises
        label = int(row['label_idx'])
        if self.transform:
            image = self.transform(image)
        return image, label


csv_ds = ImageCSVDataset('data/synthetic/train.csv', transform=transform)
print(len(csv_ds))
img, lbl = csv_ds[0]
print(img.shape, lbl)

## 4. Transforms — `torchvision.transforms.v2` (modern API)

`v2` is the current recommended API. Key upgrade: transforms now also apply to bounding boxes and masks, not just images — important for detection/segmentation tasks.

In [ ]:
# Training transforms — augmentation + normalization
train_transform = v2.Compose([
    v2.Resize((64, 64)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),     # [0,255] uint8 → [0,1] float32
    v2.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

# Test transforms — no augmentation, just resize + normalize
test_transform = v2.Compose([
    v2.Resize((64, 64)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

print('Train transform:', train_transform)
print('Test transform: ', test_transform)

In [ ]:
# Visualize augmentation effect on the same image
raw_img = Image.open(list(pathlib.Path('data/synthetic/train/red').glob('*.jpg'))[0])

aug_transform = v2.Compose([
    v2.Resize((64, 64)),
    v2.RandomHorizontalFlip(p=1.0),
    v2.RandomRotation(degrees=30),
    v2.ColorJitter(brightness=0.4, contrast=0.4),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
axes[0].imshow(raw_img.resize((64,64)))
axes[0].set_title('original')
axes[0].axis('off')
for i in range(1, 6):
    t = aug_transform(raw_img).permute(1,2,0).clamp(0,1)
    axes[i].imshow(t)
    axes[i].set_title(f'aug {i}')
    axes[i].axis('off')
plt.suptitle('Same image, different augmentations', fontsize=10)
plt.tight_layout()
plt.show()

## 5. DataLoader workers and `pin_memory`

For large datasets, loading images on the CPU is a bottleneck.  
`num_workers` spawns background processes to prefetch batches while the GPU trains.

In [ ]:
loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,       # background workers prefetch batches (0 = main process only)
    pin_memory=True,     # keep batch in pinned (page-locked) RAM for faster GPU transfer
    persistent_workers=True,  # keep workers alive between epochs (avoids spawn overhead)
)

xb, yb = next(iter(loader))
print(xb.shape, yb.shape)

## 6. Quick end-to-end check

Train a tiny model on the synthetic dataset to verify the pipeline works.

In [ ]:
import torch.nn as nn

aug_train_ds = datasets.ImageFolder('data/synthetic/train', transform=train_transform)
aug_test_ds  = datasets.ImageFolder('data/synthetic/test',  transform=test_transform)
train_loader = DataLoader(aug_train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(aug_test_ds,  batch_size=16)

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64 * 64 * 3, 64), nn.ReLU(),
    nn.Linear(64, 3)
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        criterion(model(xb), yb).backward()
        optimizer.step()

model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total   += yb.size(0)
print(f'Test accuracy: {correct/total:.4f}')

# Cleanup
shutil.rmtree('data/synthetic')